# Notebook 04 - Customer Segmentation
**Input:** `data/processed/customer_features.parquet` (5,265 customers x 38 columns)<br>
**Output:** `data/processed/customer_segments.parquet` - same customers, now with segment labels

## Why segmentation comes before predictive modeling

Predictive models (CLV, churn) tell us *what will happen*. Segmentation tells us *who these people are* - and that's what lets us act.

A segment is a label like "Champion" or "At Risk" that summarizes a customer's situation in one word.<br>
That label is what shows up in dashboards, in marketing automation rules, and in the conversation with your boss.<br>
"Predicted CLV of £247.30" is hard to act on; "Champion, predicted to stay" is immediately useful.

## Two complementary approaches
We'll do both because they tell different stories:

**1. RFM scoring (rules-based)** - Score each customer 1-5 on Recency, Frequency, Monetary using quintiles.<br>
Map score patterns to named segments (Champions, Loyal, At Risk, etc.). This is the classic marketing approach: explainable, no ML needed, defensible to any stakeholder.

**2. K-Means Clustering (data-dirven)** - Let an unsupervised algorithm find natural grouping using more features than just RFM. May surface segments rules would miss.

**Strategy:** Use RFM segment as the primary labels (because they're business-friendly), and use K-Means as a secondary cross-check to see if natural grouping agree or reveal something new.

## What this notebook is NOT

It's not modeling - there's no target variable, no train/test split, no predictions. Segmentation is **unsupervised**: we're organizing customers into groups based on their current state, not predicting their future.

## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../data/processed')
REPORTS_DIR = Path('../reports/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
features = pd.read_parquet(PROCESSED_DIR / 'customer_features.parquet')
print(f'Loaded {len(features):,} customers x {features.shape[1]} columns')

Loaded 5,256 customers x 38 columns


# PART 1 - RFM Scoring (Rules-Based Segmentation)

## 1. Compute RFM scores (1-5 quintiles)

For each of Recency, Frequency, Monetary, we score every customer 1 (worst) to 5 (best):
- **R-score**: Lower recency_days = better -> score 5 for the most recent buyers, 1 for the oldest
- **F-score**: Higher frequency = better -> score 5 for most frequent
- **M-score**: Higher monetary = better -> score 5 for biggest spenders

`qcut` splits each metric into 5 buckets containing roughly equal numbers of customers (quintiles).

In [3]:
# R-score: invert so that LOWER recency_days gives HIGHER score
features['R_score'] = pd.qcut(features['recency_days'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)

# F-score: many customers have frequency=1 which can cause duplicate-edge errors.
# Use rank to break ties uniformly before qcut.
features['F_score'] = pd.qcut(features['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

# M-score: same rank trick for safety
features['M_score'] = pd.qcut(features['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

# Combined RFM score as a string (e.g. '555' for the very best customers)
features['RFM_score'] = (
    features['R_score'].astype(str) +
    features['F_score'].astype(str) +
    features['M_score'].astype(str)
)

features[['CustomerID', 'recency_days', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score']].head(10)

,CustomerID,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_score
0,12346.0,235,3,"77,352.96",3,3,5,335
1,12347.0,39,6,"4,114.18",4,4,5,445
2,12348.0,158,4,"1,388.40",3,3,4,334
3,12349.0,317,2,"2,221.14",2,2,4,224
4,12350.0,219,1,294.40,3,1,2,312
5,12351.0,284,1,300.93,2,1,2,212
6,12352.0,171,6,985.31,3,4,3,343
7,12353.0,113,2,406.76,3,2,2,322
8,12354.0,141,1,"1,079.40",3,1,3,313
9,12355.0,123,2,947.61,3,2,3,323


In [4]:
# Sanity check: each score should have roughly 20% of customers in each level
print('R_score distribution:')
print(features['R_score'].value_counts().sort_index())
print('\nF_score distribution:')
print(features['F_score'].value_counts().sort_index())
print('\nM_score distribution:')
print(features['M_score'].value_counts().sort_index())

R_score distribution:
R_score
1    1051
2    1051
3    1051
4    1051
5    1052
Name: count, dtype: int64

F_score distribution:
F_score
1    1052
2    1051
3    1051
4    1051
5    1051
Name: count, dtype: int64

M_score distribution:
M_score
1    1052
2    1051
3    1051
4    1051
5    1051
Name: count, dtype: int64


## 2. Map score patterns to named segments
Industry-standard RFM segment mapping. The rules are based in which dimensions are strong (4-5) vs. weak (1-2)
| Segment | Logic | Strategy |
|---|---|---|
| **Champions** | R=4-5, F=4-5, M=4-5 | Reward, ask for referrals |
| **Loyal Customers** | F=3-5, R=3-5 (not Champions) | Upsell, cross-sell |
| **Potential Loyalists** | R=4-5, F=1-2 | Onboard further, build habit |
| **New Customers** | R=4-5, F=1 | Welcome, educate |
| **At Risk** | R=2-3, F=3-5, M=3-5 | Personalized win-back |
| **Cannot Lose Them** | R=1-2, F=4-5, M=4-5 | Urgent intervention |
| **Hibernating** | R=1-2, F=1-2, M=1-2 | Light re-engagement |
| **Lost** | R=1, F=1, M=1 | Likely write-off |
| **Need Attention** | Everything else | Default retention |

In [5]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    
    # Highest =value retantion priorities
    if r >= 4 and f >= 4 and m>= 4:
        return 'Champions'
    if r <= 2 and f >= 4 and m >= 4:
        return 'Cannot Lose Them'
    
    # Active and engaged
    if r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    if r >= 4 and f <= 2:
        return 'New Customers' if f == 1 else 'Potential Loyalists'

    # Lapsing customers worth saving
    if r <= 3 and f >= 3 and m >= 3:
        return 'At Risk'

    # Disengaging
    if r <= 2 and f <= 2 and m <= 2:
        return 'Hibernating' if r == 2 else 'Lost'

    return 'Need Attention'

features['segment'] = features.apply(assign_segment, axis=1)

segment_counts = features['segment'].value_counts()
print('Segment sizes:')
print(segment_counts)
print(f'\nTotal: {segment_counts.sum():,} customers')

Segment sizes:
segment
Champions              1203
Need Attention         1087
Loyal Customers         923
Lost                    665
Hibernating             414
At Risk                 364
Cannot Lose Them        216
Potential Loyalists     214
New Customers           170
Name: count, dtype: int64

Total: 5,256 customers


## 3. Profile each segment - the business-facing view
What does each segment look like? This is THE table to show your boss. It connects the abstract scores to real business meaning.

In [6]:
segment_profile = features.groupby('segment').agg(
    customers=('CustomerID', 'count'),
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_aov=('avg_order_value', 'mean'),
    total_revenue=('monetary', 'sum'),
    repurchase_rate=('target_purchased_90d', 'mean'),
    avg_future_90d_revenue=('target_revenue_90d', 'mean')
).round(2)

segment_profile['pct_of_customers'] = (segment_profile['customers'] / segment_profile['customers'].sum() * 100).round(1)
segment_profile['pct_of_revenue'] = (segment_profile['total_revenue'] / segment_profile['total_revenue'].sum() * 100).round(1)

# Reorder columns and sort by total revenue (your most valuable segment first)
segment_profile = segment_profile[[
    'customers', 'pct_of_customers', 'total_revenue', 'pct_of_revenue',
    'avg_recency', 'avg_frequency', 'avg_monetary', 'avg_aov',
    'repurchase_rate', 'avg_future_90d_revenue'
]].sort_values('total_revenue', ascending=False)

segment_profile

,customers,pct_of_customers,total_revenue,pct_of_revenue,avg_recency,avg_frequency,avg_monetary,avg_aov,repurchase_rate,avg_future_90d_revenue
segment,,,,,,,,,,
Champions,1203,22.90,"9,848,935.33",70.10,35.79,15.32,"8,186.98",459.59,0.80,"1,675.11"
Loyal Customers,923,17.60,"1,927,555.34",13.70,116.81,5.24,"2,088.36",423.31,0.53,363.25
Cannot Lose Them,216,4.10,"760,126.63",5.40,328.82,7.91,"3,519.10",440.59,0.36,270.57
Need Attention,1087,20.70,"634,345.67",4.50,238.95,1.95,583.57,392.62,0.29,335.18
At Risk,364,6.90,"412,468.23",2.90,348.93,3.23,"1,133.15",385.43,0.30,164.23
Lost,665,12.70,"152,533.57",1.10,496.94,1.11,229.37,212.28,0.10,40.01
Potential Loyalists,214,4.10,"134,994.32",1.00,46.77,1.68,630.81,360.80,0.50,316.98
Hibernating,414,7.90,"112,123.91",0.80,308.84,1.18,270.83,238.87,0.20,71.91
New Customers,170,3.20,"62,957.28",0.40,44.89,1.00,370.34,370.34,0.45,252.87
